# 🧠 Deep Learning: Emotion Classification with BiLSTM, GRU and Transformer

Proyek ini membangun model Deep Learning menggunakan arsitektur Deep Learning untuk melakukan klasifikasi emosi pada teks.

## 1. Load Data

In [28]:
import pandas as pd

df = pd.read_csv("C:/Users/arielva/pba2026-kelompok14/data/processed/emotion_dataset_preprocessed.csv")

# cek data
print(df.head())
print(df['emotion'].value_counts())

# label encoding (sorted biar konsisten)
labels = sorted(df['emotion'].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

df['label'] = df['emotion'].map(label2id)

print(label2id)

                                     text emotion
0       i m so furious with you right now   anger
1   my day is ruined because of this crap   anger
2        i feel like i m about to explode   anger
3        this is the worst day of my life   anger
4  i hate everything about this situation   anger
emotion
happiness         7797
disgust           4315
jealousy          4081
surprise          4062
gratitude         4004
relief            3964
guilt             3961
anger             3860
disappointment    3832
embarrassment     3788
anxiety           3776
pride             3772
hope              3770
loneliness        3751
excitement        3643
fear              3574
sadness           3559
confusion         3429
love              3365
frustration       3292
Name: count, dtype: int64
{'anger': 0, 'anxiety': 1, 'confusion': 2, 'disappointment': 3, 'disgust': 4, 'embarrassment': 5, 'excitement': 6, 'fear': 7, 'frustration': 8, 'gratitude': 9, 'guilt': 10, 'happiness': 11, 'hope': 12, 'jea

## 2. Tokenizer and Vocab

In [29]:
from collections import Counter

def simple_tokenizer(text):
    return text.lower().split()

counter = Counter()
for text in df['text']:
    counter.update(simple_tokenizer(text))

vocab = {"<pad>": 0, "<unk>": 1}

for word in counter:
    vocab[word] = len(vocab)

def encode(text):
    return [vocab.get(word, 1) for word in simple_tokenizer(text)]

print("Vocab size:", len(vocab))

Vocab size: 5620


## 3. Split Data

In [30]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df['label'],
    random_state=42
)

print(len(train_df), len(test_df))

71635 7960


## 4. Dataset and DataLoader

In [32]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class EmotionDataset(Dataset):
    def __init__(self, df):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = encode(self.texts[idx])
        return torch.tensor(tokens), self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts = pad_sequence(texts, batch_first=True)
    return texts, torch.tensor(labels)

train_loader = DataLoader(EmotionDataset(train_df), batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(EmotionDataset(test_df), batch_size=64, collate_fn=collate_fn)

## 5. Utility (Train + Eval)

In [34]:
import torch.nn as nn
from sklearn.metrics import classification_report

device = torch.device("cpu")

def train_model(model, train_loader, criterion, optimizer, epochs=5):
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

def evaluate_model(model, test_loader):
    model.eval()
    preds, trues = [], []

    with torch.no_grad():
        for x, y in test_loader:
            out = model(x)
            pred = out.argmax(1)

            preds.extend(pred.numpy())
            trues.extend(y.numpy())

    print(classification_report(trues, preds, target_names=labels))

## 6. Class Weight

In [35]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df['label']),
    y=df['label']
)

weights = torch.tensor(weights, dtype=torch.float)

## 7. Model 1 - BiLSTM

In [36]:
import torch.nn as nn

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_class):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, num_class)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        h = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(self.dropout(h))

## 8. Train BiLSTM

In [37]:
model_bilstm = BiLSTM(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=128,
    num_class=len(labels)
)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model_bilstm.parameters(), lr=1e-3)

train_model(model_bilstm, train_loader, criterion, optimizer, epochs=10)

Epoch 1, Loss: 1214.3337
Epoch 2, Loss: 522.4533
Epoch 3, Loss: 383.1484
Epoch 4, Loss: 306.9798
Epoch 5, Loss: 256.4427
Epoch 6, Loss: 215.3256
Epoch 7, Loss: 187.1097
Epoch 8, Loss: 170.1501
Epoch 9, Loss: 151.4644
Epoch 10, Loss: 135.0276


## 9. Evaluate BiLSTM

In [38]:
evaluate_model(model_bilstm, test_loader)

                precision    recall  f1-score   support

         anger       0.71      0.78      0.74       386
       anxiety       0.74      0.78      0.76       378
     confusion       0.89      0.91      0.90       343
disappointment       0.82      0.81      0.82       383
       disgust       0.93      0.91      0.92       432
 embarrassment       0.94      0.96      0.95       379
    excitement       0.98      0.97      0.98       364
          fear       0.84      0.77      0.80       357
   frustration       0.61      0.68      0.64       329
     gratitude       0.85      0.92      0.89       401
         guilt       0.97      0.95      0.96       396
     happiness       0.98      0.95      0.97       780
          hope       0.96      0.94      0.95       377
      jealousy       0.94      0.96      0.95       408
    loneliness       0.95      0.92      0.94       375
          love       0.94      0.96      0.95       337
         pride       0.95      0.92      0.94  

## 10. Model 2 - GRU

In [39]:
import torch.nn as nn

class GRUModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_class):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, num_class)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        _, h = self.gru(x)
        h = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(self.dropout(h))

## 11. Train GRU

In [41]:
model_gru = GRUModel(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=128,
    num_class=len(labels)
)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model_gru.parameters(), lr=1e-3)

train_model(model_gru, train_loader, criterion, optimizer, epochs=10)

Epoch 1, Loss: 1186.7748
Epoch 2, Loss: 495.6055
Epoch 3, Loss: 365.1155
Epoch 4, Loss: 289.5447
Epoch 5, Loss: 242.9086
Epoch 6, Loss: 208.1154
Epoch 7, Loss: 183.1356
Epoch 8, Loss: 162.7131
Epoch 9, Loss: 144.9025
Epoch 10, Loss: 132.4608


## 12. Evaluate GRU

In [42]:
evaluate_model(model_gru, test_loader)

                precision    recall  f1-score   support

         anger       0.73      0.73      0.73       386
       anxiety       0.72      0.75      0.74       378
     confusion       0.93      0.92      0.93       343
disappointment       0.79      0.79      0.79       383
       disgust       0.93      0.92      0.93       432
 embarrassment       0.93      0.96      0.94       379
    excitement       0.99      0.96      0.98       364
          fear       0.79      0.81      0.80       357
   frustration       0.63      0.66      0.64       329
     gratitude       0.91      0.88      0.90       401
         guilt       0.95      0.95      0.95       396
     happiness       0.98      0.97      0.97       780
          hope       0.93      0.94      0.94       377
      jealousy       0.94      0.93      0.94       408
    loneliness       0.95      0.94      0.94       375
          love       0.91      0.97      0.94       337
         pride       0.96      0.91      0.93  

## 13. Model 3 - Transformer Encoder

In [43]:
import torch
import torch.nn as nn

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_class, max_len=200):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Linear(embed_dim, num_class)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len).unsqueeze(0).expand(x.size(0), seq_len)

        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.transformer(x)

        # pooling (mean)
        x = x.mean(dim=1)

        return self.fc(self.dropout(x))

## 14. Train Transformer

In [45]:
model_transformer = TransformerModel(
    vocab_size=len(vocab),
    embed_dim=128,
    num_heads=4,
    hidden_dim=256,
    num_layers=2,
    num_class=len(labels)
)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model_transformer.parameters(), lr=5e-4)

train_model(model_transformer, train_loader, criterion, optimizer, epochs=10)

Epoch 1, Loss: 1671.3820
Epoch 2, Loss: 804.7547
Epoch 3, Loss: 616.5013
Epoch 4, Loss: 513.0479
Epoch 5, Loss: 452.5507
Epoch 6, Loss: 405.7501
Epoch 7, Loss: 366.7189
Epoch 8, Loss: 334.5410
Epoch 9, Loss: 312.7736
Epoch 10, Loss: 290.5905


## 15. Evaluate Transformer

In [46]:
evaluate_model(model_transformer, test_loader)

                precision    recall  f1-score   support

         anger       0.64      0.82      0.72       386
       anxiety       0.66      0.81      0.72       378
     confusion       0.93      0.88      0.91       343
disappointment       0.82      0.74      0.78       383
       disgust       0.93      0.90      0.92       432
 embarrassment       0.92      0.95      0.94       379
    excitement       0.97      0.98      0.97       364
          fear       0.83      0.68      0.75       357
   frustration       0.67      0.53      0.59       329
     gratitude       0.87      0.90      0.88       401
         guilt       0.95      0.92      0.94       396
     happiness       0.97      0.96      0.96       780
          hope       0.91      0.96      0.94       377
      jealousy       0.93      0.93      0.93       408
    loneliness       0.90      0.96      0.93       375
          love       0.94      0.95      0.94       337
         pride       0.93      0.90      0.91  

## Perbandingan Model (BiLSTM vs GRU vs Transformer)

| Model        | Accuracy | Macro F1 | Weighted F1 | Training Time |
|--------------|----------|----------|-------------|---------------|
| BiLSTM       | 0.89     | 0.88     | 0.89        | 5m 53s        |
| GRU          | 0.88     | 0.88     | 0.88        | 7m 09s        |
| Transformer  | 0.87     | 0.86     | 0.87        | 11m 09s       |

## Kesimpulan Model Terbaik

BiLSTM merupakan model terbaik berdasarkan hasil eksperimen ini.

## Alasan BiLSTM Terbaik

- Memiliki **accuracy tertinggi (0.89)**
- Memiliki **F1-score tertinggi (macro dan weighted)**
- **Training paling cepat** dibanding model lain
- Lebih efisien pada dataset ini dibanding GRU dan Transformer

## Performa GRU

- Performa mendekati BiLSTM
- Sedikit lebih lambat dalam training
- Tidak memberikan peningkatan signifikan dibanding BiLSTM

## Performa Transformer

- Akurasi paling rendah (0.87)
- Training paling lama (11 menit)
- Kemungkinan belum optimal untuk dataset ini (butuh tuning lebih lanjut atau data lebih besar)

In [47]:
torch.save(model_bilstm.state_dict(), "bilstm_model.pth")

In [48]:
import pickle

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

with open("labels.pkl", "wb") as f:
    pickle.dump(labels, f)